# DNS Shield — SOC Demo and Evidence Analysis

This notebook is an **offline analysis and presentation companion**. It does not make DNS decisions. Once the Docker stack is intentionally started, it calls the same API gateway used by the Go resolver and turns real event data into charts and evidence.

Before using it, read `../TEST_PLAN.md` and `../docs/SYSTEM_FLOW_AND_OPERATIONS_GUIDE.md`. Do not claim any result shown here until it comes from an actual run.

## 1. Connection configuration

The default points to the local Compose gateway. For a protected hosted demo, use an environment variable for the key instead of placing secrets inside this notebook.

In [ ]:
import os
from datetime import datetime, timezone

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests

GATEWAY_URL = os.getenv('DNS_SHIELD_GATEWAY_URL', 'http://localhost:8080').rstrip('/')
API_KEY = os.getenv('DNS_SHIELD_API_KEY', '')
HEADERS = {'X-DNS-Shield-Key': API_KEY} if API_KEY else {}
print(f'Gateway: {GATEWAY_URL}')
print('Authentication header configured:', bool(API_KEY))

In [ ]:
def api_get(path, **params):
    response = requests.get(f'{GATEWAY_URL}{path}', headers=HEADERS, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

def api_post(path, payload):
    response = requests.post(f'{GATEWAY_URL}{path}', headers=HEADERS, json=payload, timeout=10)
    response.raise_for_status()
    return response.json()

health = api_get('/health')
health

## 2. Run controlled pipeline demonstrations

These inputs are safe demonstration names. The known-bad demo indicator is local seed data. Results are stored by the platform as actual dashboard events after this cell runs.

In [ ]:
DEMO_CASES = [
    ('known-bad', 'c2.bad-demo.example'),
    ('benign', 'isro.gov.in'),
    ('dga-style', 'xq9m2kz7v4na.com'),
    ('typosquat', 'gooogle.com'),
]

results = {}
for name, domain in DEMO_CASES:
    results[name] = api_post('/v1/query', {
        'domain': domain,
        'client_ip': '172.28.0.120',
        'source': f'notebook:{name}',
    })

pd.DataFrame([
    {'case': name, 'domain': result['domain'], 'verdict': result['verdict'],
     'domain_risk': result['domain_risk'], 'device_risk': result['device_risk'],
     'confidence': result['confidence'], 'latency_ms': result['latency_ms']}
    for name, result in results.items()
])

## 3. Explainable pipeline visualization

Select one real result. This chart plots the exact stage contribution supplied by the gateway, rather than a manually invented explanation.

In [ ]:
selected_case = 'known-bad'  # Change to any key in results
selected = results[selected_case]
pipeline = pd.DataFrame(selected.get('pipeline', []))
display(pipeline[['stage', 'status', 'contribution', 'reason']])

fig = px.bar(pipeline, x='stage', y='contribution', color='status',
             hover_data=['reason'], title=f"{selected['domain']} — {selected['verdict']} XAI stage contributions")
fig.update_layout(yaxis_title='Risk contribution', xaxis_title='Pipeline stage')
fig.show()

print('Decision reasons:')
for reason in selected.get('reasons', []):
    print('-', reason)

## 4. Persisted event and verdict visualizations

In [ ]:
events = pd.DataFrame(api_get('/v1/events', limit=500))
if events.empty:
    print('No persisted events returned yet.')
else:
    events['timestamp'] = pd.to_datetime(events['timestamp'], errors='coerce', utc=True)
    display(events.head())
    fig = px.histogram(events, x='verdict', color='verdict',
                       category_orders={'verdict': ['ALLOW', 'FLAG', 'BLOCK']},
                       title='Persisted DNS security verdicts')
    fig.show()
    fig = px.scatter(events, x='timestamp', y='domain_risk', color='verdict',
                     hover_data=['domain', 'client_ip', 'device_risk'],
                     title='Domain risk over time')
    fig.show()

## 5. Hourly trend, incidents, and reputation

In [ ]:
trend = pd.DataFrame(api_get('/v1/trends', hours=24).get('points', []))
if not trend.empty:
    trend['hour'] = pd.to_datetime(trend['hour'], errors='coerce', utc=True)
    fig = go.Figure()
    fig.add_scatter(x=trend['hour'], y=trend['blocked_count'], mode='lines+markers', name='Blocked')
    fig.add_scatter(x=trend['hour'], y=trend['flagged_count'], mode='lines+markers', name='Flagged')
    fig.add_scatter(x=trend['hour'], y=trend['avg_domain_risk'], mode='lines', name='Average domain risk', yaxis='y2')
    fig.update_layout(title='24-hour security trend', yaxis_title='Event count',
                      yaxis2=dict(title='Average risk', overlaying='y', side='right'))
    fig.show()

incidents = api_get('/v1/incidents')
pd.DataFrame([{'id': item['id'], 'device': item['device'], 'severity': item['severity'],
               'timeline_entries': len(item.get('timeline', [])), 'summary': item['summary']}
              for item in incidents])

In [ ]:
# Inspect one real device/domain pair after events exist.
if not events.empty:
    example = events.iloc[0]
    device_profile = api_get(f"/v1/devices/{example['client_ip']}")
    domain_profile = api_get(f"/v1/domains/{example['domain']}")
    display(pd.DataFrame([device_profile.get('profile', {})]))
    display(pd.DataFrame([domain_profile.get('profile', {})]))
    domain_profile.get('parent_poisoning_analysis')

## 6. Operational health and evidence notes

Use these responses to prove configuration state, not to fabricate performance. For latency/QPS/resilience claims, follow the controlled steps in `TEST_PLAN.md` and keep the actual output separately.

In [ ]:
feed_health = api_get('/v1/feed-health')
model_monitoring = api_get('/v1/model-monitoring')
stats = api_get('/v1/stats')

print('Feed health')
display(pd.DataFrame(feed_health.get('feeds', [])))
print('Model monitoring')
display(pd.json_normalize(model_monitoring))
print('Verdict summary')
display(pd.DataFrame(stats.get('by_verdict', [])))

## 7. Presentation/export discipline

Only export charts after recording the date, local configuration, scenario, and event IDs that produced them. If the model is still in baseline mode, say so. If a dependency was degraded, show it. This transparency is much stronger in a security review than made-up metrics.